# ML models

Train Random Forest, LightGBM, and XGBoost on the cleaned IEEE-CIS table: baseline, feature engineering, ablations, then memory-safe SMOTE / undersampling.

Each `save_results()` call writes to `saved/ml_results.parquet`. Charts and the compact Random Forest refit are in `02_ml_models_result.ipynb`.

Kernel: `ai` (LightGBM, XGBoost, scikit-learn, imbalanced-learn).


In [1]:
import warnings
import numpy as np
import pandas as pd
import pyarrow
import fastparquet
import altair as alt
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
%matplotlib inline
pd.options.display.precision = 15
alt.renderers.enable('mimetype')
%env JOBLIB_TEMP_FOLDER=/tmp

env: JOBLIB_TEMP_FOLDER=/tmp


In [2]:
# folder_path = 'dataset/'
from pathlib import Path
import pandas as pd

# Detect environment
try:
    import google.colab

    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

# Root paths
if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"

print(f"Running on {'Google Colab' if IS_COLAB else 'Local'}")
print(f"Dataset path: {DATASET_PATH}")

Running on Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset


In [3]:
train = pd.read_parquet(f"{DATASET_PATH}/merged_train.parquet")
test = pd.read_parquet(f"{DATASET_PATH}/merged_test.parquet")

RANDOM_SEED = 42
START_DATE = "2026-01-01"

In [4]:
# Random Forest
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def top_feature_importances(model, columns, n=20):
    importances = np.asarray(model.feature_importances_, dtype=float)
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def run_random_forest(X_train, X_valid, y_train, y_valid, name="Random Forest"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    top20 = top_feature_importances(model, X_train.columns)
    print("\nTop 20 feature importances:")
    for row in top20:
        print(
            f"  {row['rank']:2d}. {row['feature']:<32s} "
            f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
        )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
        "Top20Importances": json.dumps(top20),
    }

In [5]:
# LightGBM
import json
from lightgbm import LGBMClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def top_feature_importances(model, columns, n=20):
    importances = np.asarray(model.feature_importances_, dtype=float)
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def run_lightgbm(X_train, X_valid, y_train, y_valid, name="LightGBM"):
    print(f"\n{'=' * 60}")
    print(name)
    print(f"{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")

    model = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        verbosity=-1,
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    top20 = top_feature_importances(model, X_train.columns)
    print("\nTop 20 feature importances:")
    for row in top20:
        print(
            f"  {row['rank']:2d}. {row['feature']:<32s} "
            f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
        )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
        "Top20Importances": json.dumps(top20),
    }

In [6]:
# XGBoost
import json
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)


def top_feature_importances(model, columns, n=20):
    importances = np.asarray(model.feature_importances_, dtype=float)
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def run_xgboost(X_train, X_valid, y_train, y_valid, name="XGBoost"):
    print(name)
    print(f"Features: {X_train.shape[1]}")

    model = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        random_state=RANDOM_SEED,
        n_jobs=-1,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_valid)
    y_pred_prob = model.predict_proba(X_valid)[:, 1]

    accuracy = accuracy_score(y_valid, y_pred)
    precision = precision_score(y_valid, y_pred, zero_division=0)
    recall = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_valid, y_pred_prob)
    pr_auc = average_precision_score(y_valid, y_pred_prob)
    balanced_acc = balanced_accuracy_score(y_valid, y_pred)
    mcc = matthews_corrcoef(y_valid, y_pred)
    cm = confusion_matrix(y_valid, y_pred)

    print(f"Accuracy           : {accuracy:.4f}")
    print(f"Precision          : {precision:.4f}")
    print(f"Recall             : {recall:.4f}")
    print(f"F1 Score           : {f1:.4f}")
    print(f"ROC-AUC            : {roc_auc:.4f}")
    print(f"PR-AUC             : {pr_auc:.4f}")
    print(f"Balanced Accuracy  : {balanced_acc:.4f}")
    print(f"MCC                : {mcc:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(
        classification_report(
            y_valid,
            y_pred,
            target_names=["Legitimate", "Fraud"],
            digits=4,
            zero_division=0,
        )
    )

    top20 = top_feature_importances(model, X_train.columns)
    print("\nTop 20 feature importances:")
    for row in top20:
        print(
            f"  {row['rank']:2d}. {row['feature']:<32s} "
            f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
        )

    import gc

    gc.collect()
    del model

    return {
        "Model": name,
        "Features": X_train.shape[1],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "ROC-AUC": roc_auc,
        "PR-AUC": pr_auc,
        "Balanced Accuracy": balanced_acc,
        "MCC": mcc,
        "Confusion Matrix": cm,
        "Top20Importances": json.dumps(top20),
    }

In [7]:
# DATA SPLIT / EXPERIMENT DATASETS
from itertools import combinations

train = train.sort_values("TransactionDT").reset_index(drop=True)

y = train["isFraud"]

split_idx = int(len(train) * 0.8)

y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]


# Baseline
# Original features, excluding engineered UID / UID2

baseline_cols = [
    col
    for col in train.columns
    if col not in ["isFraud", "TransactionID", "uid", "uid2"]
]

X_baseline = train[baseline_cols]

X_train_baseline = X_baseline.iloc[:split_idx]
X_valid_baseline = X_baseline.iloc[split_idx:]


# Feature Engineering
# Original + temporal + UID + UID2

feature_cols = [col for col in train.columns if col not in ["isFraud", "TransactionID"]]

X_features = train[feature_cols]

X_train_features = X_features.iloc[:split_idx]
X_valid_features = X_features.iloc[split_idx:]


# Feature-group ablation

prefix_groups = {
    "C": ("C",),
    "D": ("D",),
    "M": ("M",),
    "id": ("id_",),
    "V": ("V",),
}

group_names = list(prefix_groups.keys())

ablation_datasets = {}

for r in range(1, len(group_names) + 1):

    for combination in combinations(group_names, r):

        prefixes = tuple(
            prefix for group in combination for prefix in prefix_groups[group]
        )

        selected_cols = [col for col in baseline_cols if not col.startswith(prefixes)]

        X = train[selected_cols]

        name = "Remove " + " + ".join(combination)

        ablation_datasets[name] = (X.iloc[:split_idx], X.iloc[split_idx:])

print("Baseline features :", len(baseline_cols))
print("Feature engineering:", len(feature_cols))
print("Ablation experiments:", len(ablation_datasets))

# Reduced dataset (drop columns that were >=90% empty on the raw merge).
# Same temporal split and labels as the full table.
reduced_ablation_datasets = {}
reduced_path = DATASET_PATH / "merged_train_reduced.parquet"
if reduced_path.exists():
    train_reduced = pd.read_parquet(reduced_path)
    train_reduced = train_reduced.sort_values("TransactionDT").reset_index(drop=True)
    if len(train_reduced) != len(train):
        raise ValueError(
            f"reduced rows {len(train_reduced):,} != full train {len(train):,}"
        )

    reduced_baseline_cols = [
        c
        for c in train_reduced.columns
        if c not in ["isFraud", "TransactionID", "uid", "uid2"]
    ]
    reduced_feature_cols = [
        c for c in train_reduced.columns if c not in ["isFraud", "TransactionID"]
    ]

    X_train_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[:split_idx]
    X_valid_reduced_baseline = train_reduced[reduced_baseline_cols].iloc[split_idx:]
    X_train_reduced_features = train_reduced[reduced_feature_cols].iloc[:split_idx]
    X_valid_reduced_features = train_reduced[reduced_feature_cols].iloc[split_idx:]
    X_train_reduced = X_train_reduced_features
    X_valid_reduced = X_valid_reduced_features

    for r in range(1, len(group_names) + 1):
        for combination in combinations(group_names, r):
            prefixes = tuple(
                prefix for group in combination for prefix in prefix_groups[group]
            )
            selected_cols = [
                col for col in reduced_baseline_cols if not col.startswith(prefixes)
            ]
            name = "Reduced - Remove " + " + ".join(combination)
            X = train_reduced[selected_cols]
            reduced_ablation_datasets[name] = (
                X.iloc[:split_idx],
                X.iloc[split_idx:],
            )

    print("Reduced baseline features :", len(reduced_baseline_cols))
    print("Reduced feature engineering:", len(reduced_feature_cols))
    print("Reduced ablation experiments:", len(reduced_ablation_datasets))
else:
    X_train_reduced = X_valid_reduced = None
    X_train_reduced_baseline = X_valid_reduced_baseline = None
    X_train_reduced_features = X_valid_reduced_features = None
    print("No merged_train_reduced.parquet yet — run 01_clean_dataset.ipynb to create it.")

Baseline features : 437
Feature engineering: 439
Ablation experiments: 31
Reduced baseline features : 425
Reduced feature engineering: 427
Reduced ablation experiments: 31


In [8]:
# ALL ML EXPERIMENTS
rf_results = []
lgb_results = []
xgb_results = []

In [9]:
import json


def save_results():

    def prepare(results, model_type):
        df = pd.DataFrame(results).copy()
        if df.empty:
            return df

        # Convert model objects to model names
        if "model" in df.columns:
            df["model"] = df["model"].apply(lambda x: type(x).__name__)

        # Convert confusion matrix to TN/FP/FN/TP
        if "Confusion Matrix" in df.columns:
            cm = df.pop("Confusion Matrix").tolist()

            df["TN"] = [x[0][0] for x in cm]
            df["FP"] = [x[0][1] for x in cm]
            df["FN"] = [x[1][0] for x in cm]
            df["TP"] = [x[1][1] for x in cm]

        if "Top20Importances" in df.columns:
            df["Top20Importances"] = df["Top20Importances"].apply(
                lambda x: x if isinstance(x, str) else json.dumps(x if x == x else [])
            )

        return df.assign(ModelType=model_type)

    frames = [
        df
        for df in [
            prepare(rf_results, "RandomForest"),
            prepare(lgb_results, "LightGBM"),
            prepare(xgb_results, "XGBoost"),
        ]
        if not df.empty
    ]
    if not frames:
        return

    new_df = pd.concat(frames, ignore_index=True)
    path = f"{SAVED_PATH}/ml_results.parquet"
    if Path(path).exists():
        old = pd.read_parquet(path)
        if "Model" in new_df.columns and "Model" in old.columns:
            old = old[~old["Model"].isin(new_df["Model"])]
        new_df = pd.concat([old, new_df], ignore_index=True)

    new_df.to_parquet(path, index=False)
    print(f"saved {path}  n={len(new_df)}")

In [10]:
# 1. BASELINE - RF

rf_results.append(
    run_random_forest(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "RF - Baseline"
    )
)
save_results()


# 2. FEATURE ENGINEERING

rf_results.append(
    run_random_forest(
        X_train_features, X_valid_features, y_train, y_valid, "RF - Feature Engineering"
    )
)
save_results()

if X_train_reduced_features is not None:
    rf_results.append(
        run_random_forest(
            X_train_reduced_features,
            X_valid_reduced_features,
            y_train,
            y_valid,
            "RF - Reduced Feature Engineering",
        )
    )
    save_results()



RF - Baseline
Features: 437
Accuracy           : 0.9740
Precision          : 0.8058
Recall             : 0.3206
F1 Score           : 0.4587
ROC-AUC            : 0.9056
PR-AUC             : 0.5279
Balanced Accuracy  : 0.6589
MCC                : 0.4986

Confusion Matrix:
[[113730    314]
 [  2761   1303]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9763    0.9972    0.9867    114044
       Fraud     0.8058    0.3206    0.4587      4064

    accuracy                         0.9740    118108
   macro avg     0.8911    0.6589    0.7227    118108
weighted avg     0.9704    0.9740    0.9685    118108


Top 20 feature importances:
   1. TransactionAmt                   0.025819  (2.58%)
   2. C13                              0.024890  (2.49%)
   3. TransactionDT                    0.023175  (2.32%)
   4. card1                            0.020867  (2.09%)
   5. C14                              0.020529  (2.05%)
   6. card2                 

In [11]:
# 1. BASELINE - LGBM
lgb_results.append(
    run_lightgbm(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "LightGBM - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
lgb_results.append(
    run_lightgbm(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "LightGBM - Feature Engineering",
    )
)
save_results()

if X_train_reduced_features is not None:
    lgb_results.append(
        run_lightgbm(
            X_train_reduced_features,
            X_valid_reduced_features,
            y_train,
            y_valid,
            "LightGBM - Reduced Feature Engineering",
        )
    )
    save_results()


LightGBM - Baseline
Features: 437
Accuracy           : 0.9008
Precision          : 0.2180
Recall             : 0.7274
F1 Score           : 0.3355
ROC-AUC            : 0.9057
PR-AUC             : 0.5203
Balanced Accuracy  : 0.8172
MCC                : 0.3627

Confusion Matrix:
[[103440  10604]
 [  1108   2956]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9894    0.9070    0.9464    114044
       Fraud     0.2180    0.7274    0.3355      4064

    accuracy                         0.9008    118108
   macro avg     0.6037    0.8172    0.6409    118108
weighted avg     0.9629    0.9008    0.9254    118108


Top 20 feature importances:
   1. card1                            984.000000  (6.56%)
   2. card2                            718.000000  (4.79%)
   3. addr1                            579.000000  (3.86%)
   4. TransactionAmt                   544.000000  (3.63%)
   5. TransactionDT                    519.000000  (3.46%)
   6. C13   

In [12]:
# 1. BASELINE - XGB
xgb_results.append(
    run_xgboost(
        X_train_baseline, X_valid_baseline, y_train, y_valid, "XGBoost - Baseline"
    )
)
save_results()

# 2. FEATURE ENGINEERING
xgb_results.append(
    run_xgboost(
        X_train_features,
        X_valid_features,
        y_train,
        y_valid,
        "XGBoost - Feature Engineering",
    )
)
save_results()

if X_train_reduced_features is not None:
    xgb_results.append(
        run_xgboost(
            X_train_reduced_features,
            X_valid_reduced_features,
            y_train,
            y_valid,
            "XGBoost - Reduced Feature Engineering",
        )
    )
    save_results()

XGBoost - Baseline
Features: 437
Accuracy           : 0.9110
Precision          : 0.2365
Recall             : 0.7124
F1 Score           : 0.3551
ROC-AUC            : 0.9017
PR-AUC             : 0.5170
Balanced Accuracy  : 0.8152
MCC                : 0.3770

Confusion Matrix:
[[104698   9346]
 [  1169   2895]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9890    0.9180    0.9522    114044
       Fraud     0.2365    0.7124    0.3551      4064

    accuracy                         0.9110    118108
   macro avg     0.6127    0.8152    0.6536    118108
weighted avg     0.9631    0.9110    0.9316    118108


Top 20 feature importances:
   1. V258                             0.167781  (16.78%)
   2. V70                              0.083564  (8.36%)
   3. V91                              0.043781  (4.38%)
   4. V294                             0.043437  (4.34%)
   5. V201                             0.030291  (3.03%)
   6. C8               

In [13]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():

    rf_results.append(
        run_random_forest(X_train, X_valid, y_train, y_valid, f"RF - {name}")
    )
    save_results()

for name, (X_train, X_valid) in reduced_ablation_datasets.items():

    rf_results.append(
        run_random_forest(X_train, X_valid, y_train, y_valid, f"RF - {name}")
    )
    save_results()


RF - Remove C
Features: 423
Accuracy           : 0.9729
Precision          : 0.7940
Recall             : 0.2864
F1 Score           : 0.4210
ROC-AUC            : 0.8916
PR-AUC             : 0.4879
Balanced Accuracy  : 0.6419
MCC                : 0.4672

Confusion Matrix:
[[113742    302]
 [  2900   1164]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9751    0.9974    0.9861    114044
       Fraud     0.7940    0.2864    0.4210      4064

    accuracy                         0.9729    118108
   macro avg     0.8846    0.6419    0.7035    118108
weighted avg     0.9689    0.9729    0.9667    118108


Top 20 feature importances:
   1. TransactionAmt                   0.031469  (3.15%)
   2. TransactionDT                    0.026671  (2.67%)
   3. card1                            0.023728  (2.37%)
   4. card2                            0.022282  (2.23%)
   5. addr1                            0.020536  (2.05%)
   6. DT_day                

In [14]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    lgb_results.append(
        run_lightgbm(X_train, X_valid, y_train, y_valid, f"LightGBM - {name}")
    )

    save_results()

for name, (X_train, X_valid) in reduced_ablation_datasets.items():
    lgb_results.append(
        run_lightgbm(X_train, X_valid, y_train, y_valid, f"LightGBM - {name}")
    )
    save_results()


LightGBM - Remove C
Features: 423
Accuracy           : 0.8939
Precision          : 0.2027
Recall             : 0.7104
F1 Score           : 0.3155
ROC-AUC            : 0.8941
PR-AUC             : 0.4810
Balanced Accuracy  : 0.8054
MCC                : 0.3419

Confusion Matrix:
[[102691  11353]
 [  1177   2887]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9887    0.9005    0.9425    114044
       Fraud     0.2027    0.7104    0.3155      4064

    accuracy                         0.8939    118108
   macro avg     0.5957    0.8054    0.6290    118108
weighted avg     0.9616    0.8939    0.9209    118108


Top 20 feature importances:
   1. card1                            898.000000  (5.99%)
   2. card2                            790.000000  (5.27%)
   3. TransactionAmt                   640.000000  (4.27%)
   4. addr1                            635.000000  (4.23%)
   5. TransactionDT                    525.000000  (3.50%)
   6. D15   

In [15]:
# 3. ALL ABLATION EXPERIMENTS
for name, (X_train, X_valid) in ablation_datasets.items():
    xgb_results.append(
        run_xgboost(
            X_train, X_valid,
            y_train, y_valid,
            f"XGBoost - {name}"
        )
    )

    save_results()

for name, (X_train, X_valid) in reduced_ablation_datasets.items():
    xgb_results.append(
        run_xgboost(
            X_train, X_valid,
            y_train, y_valid,
            f"XGBoost - {name}"
        )
    )
    save_results()

XGBoost - Remove C
Features: 423
Accuracy           : 0.8971
Precision          : 0.2075
Recall             : 0.7055
F1 Score           : 0.3206
ROC-AUC            : 0.8933
PR-AUC             : 0.4763
Balanced Accuracy  : 0.8047
MCC                : 0.3456

Confusion Matrix:
[[103092  10952]
 [  1197   2867]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate     0.9885    0.9040    0.9444    114044
       Fraud     0.2075    0.7055    0.3206      4064

    accuracy                         0.8971    118108
   macro avg     0.5980    0.8047    0.6325    118108
weighted avg     0.9616    0.8971    0.9229    118108


Top 20 feature importances:
   1. V258                             0.100074  (10.01%)
   2. V70                              0.079869  (7.99%)
   3. addr2                            0.051118  (5.11%)
   4. V91                              0.050027  (5.00%)
   5. V294                             0.046306  (4.63%)
   6. V201             

The reduced table (`merged_train_reduced.parquet`) is included in **# 2. FEATURE ENGINEERING** (`Reduced Feature Engineering`) and **# 3. ALL ABLATION EXPERIMENTS** (`Reduced - Remove …`) for RF, LightGBM, and XGBoost. Each `save_results()` call writes directly to `saved/ml_results.parquet` for `02_ml_models_result.ipynb`.


In [16]:
print(
    "Experimental runs only. Analysis lives in 02_ml_models_result.ipynb, "
    "which reads saved/ml_results.parquet."
)
print("Reduced FE ready:", X_train_reduced_features is not None)
print("Reduced ablations:", len(reduced_ablation_datasets))


Experimental runs only. Analysis lives in 02_ml_models_result.ipynb, which reads saved/ml_results.parquet.
Reduced FE ready: True
Reduced ablations: 31


## Sampling experiments (SMOTE / undersampling)

These cells do **not** retrain baseline or ablations. They fit RF, LightGBM, and XGBoost on a few feature sets with:
- SMOTE (undersample majority first, then oversample — avoids the 1.4 GiB OOM)
- Random undersampling
- SMOTE + undersampling

Each run **appends** to `saved/ml_results.parquet` (same schema, including top-20 importances). Already-saved model names are skipped, so you can re-run after a crash.

Analysis is in `02_ml_models_result.ipynb`.


In [1]:
import gc
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
pd.options.display.precision = 4
RANDOM_SEED = 42

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

ROOT = Path("/content/drive/MyDrive/minor-thesis") if IS_COLAB else Path.cwd()
DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
RESULTS_PATH = SAVED_PATH / "ml_results.parquet"

print("Environment:", "Colab" if IS_COLAB else "Local")
print("Results:", RESULTS_PATH)
print("Existing rows:", len(pd.read_parquet(RESULTS_PATH)) if RESULTS_PATH.exists() else 0)

Environment: Local
Results: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet
Existing rows: 594


In [2]:
# Same temporal 80/20 split as the training cells above
train = pd.read_parquet(DATASET_PATH / "merged_train.parquet")
train = train.sort_values("TransactionDT").reset_index(drop=True)
split_idx = int(len(train) * 0.8)
y = train["isFraud"]
y_train, y_valid = y.iloc[:split_idx], y.iloc[split_idx:]

baseline_cols = [
    c for c in train.columns if c not in ["isFraud", "TransactionID", "uid", "uid2"]
]
feature_cols = [c for c in train.columns if c not in ["isFraud", "TransactionID"]]

datasets = {
    "Baseline": (
        train[baseline_cols].iloc[:split_idx],
        train[baseline_cols].iloc[split_idx:],
    ),
    "Feature Engineering": (
        train[feature_cols].iloc[:split_idx],
        train[feature_cols].iloc[split_idx:],
    ),
}

id_v = [c for c in baseline_cols if not c.startswith(("id_", "V"))]
d_id_v = [c for c in baseline_cols if not c.startswith(("D", "id_", "V"))]
datasets["Remove id + V"] = (
    train[id_v].iloc[:split_idx],
    train[id_v].iloc[split_idx:],
)
datasets["Remove D + id + V"] = (
    train[d_id_v].iloc[:split_idx],
    train[d_id_v].iloc[split_idx:],
)

reduced_path = DATASET_PATH / "merged_train_reduced.parquet"
if reduced_path.exists():
    reduced = pd.read_parquet(reduced_path).sort_values("TransactionDT").reset_index(drop=True)
    if len(reduced) != len(train):
        raise ValueError(f"reduced rows {len(reduced):,} != {len(train):,}")
    red_base = [c for c in reduced.columns if c not in ["isFraud", "TransactionID", "uid", "uid2"]]
    red_fe = [c for c in reduced.columns if c not in ["isFraud", "TransactionID"]]
    datasets["Reduced Baseline"] = (
        reduced[red_base].iloc[:split_idx],
        reduced[red_base].iloc[split_idx:],
    )
    datasets["Reduced Feature Engineering"] = (
        reduced[red_fe].iloc[:split_idx],
        reduced[red_fe].iloc[split_idx:],
    )
    del reduced

print("Train/valid:", len(y_train), len(y_valid), "fraud rate", f"{y_train.mean():.4f}")
for name, (X_tr, _) in datasets.items():
    print(f"  {name}: {X_tr.shape[1]} features")


Train/valid: 472432 118108 fraud rate 0.0351
  Baseline: 437 features
  Feature Engineering: 439 features
  Remove id + V: 60 features
  Remove D + id + V: 38 features
  Reduced Baseline: 425 features
  Reduced Feature Engineering: 427 features


In [3]:
def top_feature_importances(model, columns, n=20):
    importances = np.asarray(model.feature_importances_, dtype=float)
    total = float(importances.sum())
    n = min(int(n), len(importances))
    order = np.argsort(importances)[::-1][:n]
    rows = []
    for rank, idx in enumerate(order, start=1):
        value = float(importances[idx])
        pct = (value / total * 100.0) if total > 0 else 0.0
        rows.append(
            {
                "rank": int(rank),
                "feature": str(columns[idx]),
                "importance": value,
                "importance_pct": pct,
            }
        )
    return rows


def evaluate(model, X_train, X_valid, y_train, y_valid, name):
    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print(f"Features: {X_train.shape[1]}")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)
    y_prob = model.predict_proba(X_valid)[:, 1]
    cm = confusion_matrix(y_valid, y_pred)
    top20 = top_feature_importances(model, X_train.columns)
    print(f"ROC-AUC : {roc_auc_score(y_valid, y_prob):.4f}")
    print(f"PR-AUC  : {average_precision_score(y_valid, y_prob):.4f}")
    print(f"F1      : {f1_score(y_valid, y_pred, zero_division=0):.4f}")
    print("\nTop 20 feature importances:")
    for row in top20:
        print(
            f"  {row['rank']:2d}. {row['feature']:<32s} "
            f"{row['importance']:.6f}  ({row['importance_pct']:.2f}%)"
        )
    print(classification_report(y_valid, y_pred, target_names=["Legitimate", "Fraud"], digits=4, zero_division=0))
    del model
    gc.collect()
    return {
        "Model": name,
        "Features": int(X_train.shape[1]),
        "Accuracy": float(accuracy_score(y_valid, y_pred)),
        "Precision": float(precision_score(y_valid, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_valid, y_pred, zero_division=0)),
        "F1": float(f1_score(y_valid, y_pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_valid, y_prob)),
        "PR-AUC": float(average_precision_score(y_valid, y_prob)),
        "Balanced Accuracy": float(balanced_accuracy_score(y_valid, y_pred)),
        "MCC": float(matthews_corrcoef(y_valid, y_pred)),
        "TN": int(cm[0, 0]),
        "FP": int(cm[0, 1]),
        "FN": int(cm[1, 0]),
        "TP": int(cm[1, 1]),
        "Top20Importances": json.dumps(top20),
    }


def run_random_forest(X_train, X_valid, y_train, y_valid, name):
    return evaluate(
        RandomForestClassifier(
            n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1
        ),
        X_train, X_valid, y_train, y_valid, name,
    )


def run_lightgbm(X_train, X_valid, y_train, y_valid, name):
    return evaluate(
        LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=-1,
        ),
        X_train, X_valid, y_train, y_valid, name,
    )


def run_xgboost(X_train, X_valid, y_train, y_valid, name):
    n_pos = max(int((y_train == 1).sum()), 1)
    n_neg = int((y_train == 0).sum())
    return evaluate(
        XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=6,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            eval_metric="logloss",
            scale_pos_weight=n_neg / n_pos,
        ),
        X_train, X_valid, y_train, y_valid, name,
    )


def save_results(rows):
    new_df = pd.DataFrame(rows)
    if new_df.empty:
        return
    if RESULTS_PATH.exists():
        old = pd.read_parquet(RESULTS_PATH)
        old = old[~old["Model"].isin(new_df["Model"])]
        new_df = pd.concat([old, new_df], ignore_index=True)
    RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    new_df.to_parquet(RESULTS_PATH, index=False)
    print(f"saved {RESULTS_PATH}  n={len(new_df)}")

In [4]:
def iter_sampling_variants(X_train, y_train):
    """One variant at a time. Undersample before SMOTE so we never allocate 439k x 428 float64."""
    cols = list(X_train.columns)
    X_np = np.asarray(X_train, dtype=np.float32)
    y_np = np.asarray(y_train)

    X_s, y_s = Pipeline(
        [
            ("under", RandomUnderSampler(sampling_strategy=0.2, random_state=RANDOM_SEED)),
            ("smote", SMOTE(sampling_strategy=1.0, random_state=RANDOM_SEED)),
        ]
    ).fit_resample(X_np, y_np)
    yield "SMOTE", pd.DataFrame(X_s, columns=cols), pd.Series(y_s, name="isFraud")
    del X_s, y_s
    gc.collect()

    X_u, y_u = RandomUnderSampler(random_state=RANDOM_SEED).fit_resample(X_np, y_np)
    yield "Undersampling", pd.DataFrame(X_u, columns=cols), pd.Series(y_u, name="isFraud")
    del X_u, y_u
    gc.collect()

    X_su, y_su = Pipeline(
        [
            ("under", RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_SEED)),
            ("smote", SMOTE(sampling_strategy=0.5, random_state=RANDOM_SEED)),
        ]
    ).fit_resample(X_np, y_np)
    yield "SMOTE + Undersampling", pd.DataFrame(X_su, columns=cols), pd.Series(y_su, name="isFraud")
    del X_su, y_su, X_np, y_np
    gc.collect()

In [5]:
# Append sampling runs to ml_results.parquet
existing = (
    set(pd.read_parquet(RESULTS_PATH)["Model"]) if RESULTS_PATH.exists() else set()
)
print(f"Already saved model names: {len(existing)}")

runners = [
    ("RF", run_random_forest, "RandomForest"),
    ("LightGBM", run_lightgbm, "LightGBM"),
    ("XGBoost", run_xgboost, "XGBoost"),
]

for feature_name, (X_tr, X_va) in datasets.items():
    for sampling_name, X_sampled, y_sampled in iter_sampling_variants(X_tr, y_train):
        experiment_name = f"{feature_name} - {sampling_name}"
        print(f"\n===== {experiment_name} =====")
        print(f"Train samples: {len(y_sampled):,} | Fraud rate: {y_sampled.mean():.4f}")

        for prefix, runner, model_type in runners:
            name = f"{prefix} - {experiment_name}"
            if name in existing:
                print(f"skip {name} (already in ml_results.parquet)")
                continue
            row = runner(X_sampled, X_va, y_sampled, y_valid, name)
            row["ModelType"] = model_type
            save_results([row])
            existing.add(name)

        del X_sampled, y_sampled
        gc.collect()

print("\nDone. Sampling rows in ml_results.parquet:")
all_results = pd.read_parquet(RESULTS_PATH)
mask = all_results["Model"].str.contains("SMOTE|Undersampling", regex=True)
print(all_results.loc[mask, ["Model", "Features", "ROC-AUC", "PR-AUC", "F1"]].to_string(index=False))

Already saved model names: 594

===== Baseline - SMOTE =====
Train samples: 165,990 | Fraud rate: 0.5000
skip RF - Baseline - SMOTE (already in ml_results.parquet)
skip LightGBM - Baseline - SMOTE (already in ml_results.parquet)
skip XGBoost - Baseline - SMOTE (already in ml_results.parquet)

===== Baseline - Undersampling =====
Train samples: 33,198 | Fraud rate: 0.5000
skip RF - Baseline - Undersampling (already in ml_results.parquet)
skip LightGBM - Baseline - Undersampling (already in ml_results.parquet)
skip XGBoost - Baseline - Undersampling (already in ml_results.parquet)

===== Baseline - SMOTE + Undersampling =====
Train samples: 248,985 | Fraud rate: 0.3333
skip RF - Baseline - SMOTE + Undersampling (already in ml_results.parquet)
skip LightGBM - Baseline - SMOTE + Undersampling (already in ml_results.parquet)
skip XGBoost - Baseline - SMOTE + Undersampling (already in ml_results.parquet)

===== Feature Engineering - SMOTE =====
Train samples: 165,990 | Fraud rate: 0.5000
ski